In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf
from pyspark.sql.types import StringType, IntegerType
import time


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (1, "Alice", "Engineering", 75000, 25),
    (2, "Bob", "Marketing", 60000, 30),
    (3, "Charlie", "Engineering", 80000, 35),
    (4, "David", "Sales", 65000, 28),
    (5, "Eve", "Marketing", 70000, 32),
    (6, "Frank", "Engineering", 85000, 29),
    (7, "Grace", "Sales", 68000, 31),
    (8, "Hannah", "Marketing", 72000, 27),
    (9, "Ian", "Engineering", 90000, 40),
    (10, "Jack", "HR", 58000, 26),
    (11, "Karen", "Finance", 76000, 34),
    (12, "Leo", "Engineering", 82000, 33),
    (13, "Mia", "Sales", 67000, 29),
    (14, "Nathan", "Marketing", 71000, 36),
    (15, "Olivia", "HR", 62000, 28),
    (16, "Paul", "Finance", 78000, 41),
    (17, "Quinn", "Engineering", 87000, 38),
    (18, "Rachel", "Sales", 69000, 30),
    (19, "Steve", "Marketing", 74000, 35),
    (20, "Tina", "Engineering", 92000, 42)
]
columns = ["Id", "Name", "Department", "Salary", "Age"]
df = spark.createDataFrame(data, columns);

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 05:47:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_select=df.select("Name", "Department", "Salary")

In [ ]:
df_select.show()


+-------+-----------+------+
|   Name| Department|Salary|
+-------+-----------+------+
|  Alice|Engineering| 75000|
|    Bob|  Marketing| 60000|
|Charlie|Engineering| 80000|
|  David|      Sales| 65000|
|    Eve|  Marketing| 70000|
|  Frank|Engineering| 85000|
|  Grace|      Sales| 68000|
| Hannah|  Marketing| 72000|
|    Ian|Engineering| 90000|
|   Jack|         HR| 58000|
|  Karen|    Finance| 76000|
|    Leo|Engineering| 82000|
|    Mia|      Sales| 67000|
| Nathan|  Marketing| 71000|
| Olivia|         HR| 62000|
|   Paul|    Finance| 78000|
|  Quinn|Engineering| 87000|
| Rachel|      Sales| 69000|
|  Steve|  Marketing| 74000|
|   Tina|Engineering| 92000|
+-------+-----------+------+



In [ ]:
print("Hello")

In [4]:
print("Before Partition:", df.rdd.getNumPartitions())

Before Partition: 2


In [5]:
df.write.mode("overwrite").csv("output/employeesTable", header=True)

In [6]:
df_repartitioned=df.repartition(10)

In [7]:
print("After Partition:", df_repartitioned.rdd.getNumPartitions())

After Partition: 10


In [8]:
df_repartitioned.write.mode("overwrite").csv("output/employeesTable1", header=True)

In [9]:
df_coalesced=df_repartitioned.coalesce(2)

In [10]:
print("After Coalesced:", df_coalesced.rdd.getNumPartitions())

After Coalesced: 2


26/06/13 05:47:42 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [19]:
numdata=spark.range(0, 1000000).withColumn("value", col("id")%1000)

In [20]:
numdata.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [13]:
print("Before Partition:", numdata.rdd.getNumPartitions())

Before Partition: 2


In [14]:
numdata.write.mode("overwrite").csv("output/numdata", header=True)

In [15]:
num_rep=numdata.repartition(10)

In [16]:
print("After Partition:", num_rep.rdd.getNumPartitions())

[Stage 10:>                                                         (0 + 2) / 2]

After Partition: 10


In [17]:
num_rep.write.mode("overwrite").csv("output/numdata1", header=True)

In [ ]:
#------------------------------------------------------------------------------------------------------------------------------

In [ ]:
#------------------------------------------------------------------------------------------------------------------------------

In [21]:
optimized_df=numdata.filter(col("value")>500).filter(col("id")<5000000).select("id", "value")

In [22]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#45L, (id#45L % 1000) AS value#47L]
+- *(1) Filter (((id#45L % 1000) > 500) AND (id#45L < 5000000))
   +- *(1) Range (0, 1000000, step=1, splits=2)




In [24]:
import time
start_time=time.time()
count_uncached=optimized_df.count()
end_time=time.time()
print(f"1. Optimized execution | count:{count_uncached} | time: {round(end_time-start_time, 4)} sec")

1. Optimized execution | count:499000 | time: 0.657 sec


In [25]:
optimized_df.cache()

DataFrame[id: bigint, value: bigint]

In [26]:
import time
start_time=time.time()
count_uncached=optimized_df.count()
end_time=time.time()
print(f"2. Optimized execution | count:{count_uncached} | time: {round(end_time-start_time, 4)} sec")

[Stage 18:=============================>                            (1 + 1) / 2]

2. Optimized execution | count:499000 | time: 1.017 sec


In [27]:
import time
start_time=time.time()
count_uncached=optimized_df.count()
end_time=time.time()
print(f"3. Optimized execution | count:{count_uncached} | time: {round(end_time-start_time, 4)} sec")

3. Optimized execution | count:499000 | time: 0.1507 sec
